# Mask2Former COCO Training on Kaggle

This notebook provides the setup to train Mask2Former (Swin-Tiny Backbone) on the COCO dataset in a Kaggle environment.
Kaggle environments typically store datasets in `/kaggle/input/` (read-only) and allow writing to `/kaggle/working/`.

In [1]:
!nvidia-smi

Tue May 26 11:08:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Environment Setup
Install Detectron2 and Mask2Former prerequisites. Kaggle usually comes with a recent version of PyTorch and CUDA pre-installed. Mask2Former requires building a custom CUDA kernel (`MSDeformAttn`).

In [2]:
import os
import sys
import subprocess
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
os.chdir(WORKING_DIR)

In [3]:
%%bash
# Install Detectron2
python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

# Clone Mask2Former if not already in the workspace
if [ ! -d "Mask2Former" ]; then
    git clone https://github.com/facebookresearch/Mask2Former.git
fi
cd Mask2Former
pip install -r requirements.txt

# The requirements.txt installs timm==0.3.2 which breaks on PyTorch 2.x
# We must upgrade timm to fix the missing torch._six error
pip install "timm>=0.9.16"

# Patch MSDeformAttn for PyTorch >= 1.11 compatibility
# 1. Fix .type().is_cuda() -> .is_cuda() for all tensors
sed -i 's/\.type()\.is_cuda()/.is_cuda()/g' mask2former/modeling/pixel_decoder/ops/src/cuda/ms_deform_attn_cuda.cu
sed -i 's/\.type()\.is_cuda()/.is_cuda()/g' mask2former/modeling/pixel_decoder/ops/src/cpu/ms_deform_attn_cpu.cpp
sed -i 's/\.type()\.is_cuda()/.is_cuda()/g' mask2former/modeling/pixel_decoder/ops/src/ms_deform_attn.h

# 2. Fix value.type() -> value.scalar_type() in AT_DISPATCH_FLOATING_TYPES
sed -i 's/value\.type()/value.scalar_type()/g' mask2former/modeling/pixel_decoder/ops/src/cuda/ms_deform_attn_cuda.cu
sed -i 's/value\.type()/value.scalar_type()/g' mask2former/modeling/pixel_decoder/ops/src/cpu/ms_deform_attn_cpu.cpp

# Compile the CUDA kernel for MSDeformAttn
cd mask2former/modeling/pixel_decoder/ops
sh make.sh

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-o2vrzdbp
  Resolved https://github.com/facebookresearch/detectron2.git to commit e0ec4e189d438848521aee7926f9900e114229f5
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 5.5 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x86_64.whl size=7111502 sha256=f7ff088db9813cefc3c4cb1db20ec6690a98724057de8a88805054566b38bce7
  Stored in directory: /tmp/pip-ephem-wheel-cache-cpju5h_8/wheels/d3/6e/bd/1969578f1456a6be2d6f083da65c669f450b23b8f3d1ac14c1
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=fe455bbc88ec9e4732ffc307aa711c2cf27d7

  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-o2vrzdbp
Cloning into 'Mask2Former'...
W0526 11:11:19.159000 335 torch/utils/cpp_extension.py:535] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.8
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install

## 2. Dataset Preparation
Detectron2 relies on the `DETECTRON2_DATASETS` environment variable or looks for a `datasets` folder in the current directory. We will symlink the Kaggle COCO dataset to `Mask2Former/datasets/coco`.

In [8]:
# Set this path to where the COCO dataset is located in the Kaggle input.
# e.g., "/kaggle/input/coco-2017-dataset/coco2017"
KAGGLE_COCO_PATH = Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017")
M2F_DIR = WORKING_DIR / "Mask2Former"
M2F_DATASETS = M2F_DIR / "datasets"
M2F_COCO = M2F_DATASETS / "coco"

if not KAGGLE_COCO_PATH.exists():
    print(f"Dataset not found at {KAGGLE_COCO_PATH}. Please adjust the path.")
else:
    M2F_DATASETS.mkdir(parents=True, exist_ok=True)
    if not M2F_COCO.exists():
        # Symlink Kaggle dataset to Mask2Former expected datasets/coco format
        os.symlink(str(KAGGLE_COCO_PATH), str(M2F_COCO))
        print(f"Symlinked {KAGGLE_COCO_PATH} -> {M2F_COCO}")

Verify the dataset structure matches Detectron2 expectations. For COCO, Detectron2 expects:
```
coco/
  annotations/
    instances_train2017.json
    instances_val2017.json
  train2017/
  val2017/
```

## 3. Pre-trained Backbone Weights (Optional)
To finetune using the Swin-Tiny backbone, you might need to download the ImageNet pre-trained weights if it doesn't automatically download, or you can use the fully pre-trained Mask2Former weights.

In [14]:
import os

os.chdir(M2F_DIR)
weights_dir = M2F_DIR / "weights"
weights_dir.mkdir(exist_ok=True)

swin_pth_path = weights_dir / "swin_tiny_patch4_window7_224.pth"
swin_pkl_path = weights_dir / "swin_tiny_patch4_window7_224.pkl"

if not swin_pkl_path.exists():
    print("Downloading base Swin-Tiny ImageNet weights...")
    !wget https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth -O {swin_pth_path}
    
    print("Converting weights to Detectron2 format...")
    !python tools/convert-pretrained-swin-model-to-d2.py {swin_pth_path} {swin_pkl_path}
else:
    print(f"Base weights already exist at {swin_pkl_path}")

--2026-05-26 11:24:28--  https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/357198522/fd006b80-9bd3-11eb-8445-769d89efab4e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-26T12%3A09%3A41Z&rscd=attachment%3B+filename%3Dswin_tiny_patch4_window7_224.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-26T11%3A09%3A39Z&ske=2026-05-26T12%3A09%3A41Z&sks=b&skv=2018-11-09&sig=Dg4PETgeflv%2BZsx58mRWAgIpKDkA92keYKcM%2FVndPIE%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3OTc5ODI2OCwibmJmIjoxNzc5Nzk0NjY4LCJw

## 4. Run Training
We can now invoke `train_net.py` for Mask2Former with Swin-Tiny backbone for Instance Segmentation on COCO.

In [16]:
# The model setup uses 8 GPUs by default. We configure for 1 or 2 GPUs available on Kaggle
NUM_GPUS = 2 

config_file = "configs/coco/instance-segmentation/swin/maskformer2_swin_tiny_bs16_50ep.yaml"
output_dir = WORKING_DIR / "output"

# We use -W ignore to hide all the PyTorch and Timm deprecation warnings.
# We override IMS_PER_BATCH to 4 (2 images per GPU instead of 8) and shrink the max image scale.
!python -W ignore train_net.py \
    --num-gpus {NUM_GPUS} \
    --config-file {config_file} \
    MODEL.WEIGHTS "{swin_pkl_path}" \
    OUTPUT_DIR "{output_dir}" \
    SOLVER.IMS_PER_BATCH 4 \
    INPUT.MIN_SIZE_TRAIN "(640,)" \
    INPUT.MAX_SIZE_TRAIN 640 \
    INPUT.MIN_SIZE_TEST 640 \
    INPUT.MAX_SIZE_TEST 640

Command Line Args: Namespace(config_file='configs/coco/instance-segmentation/swin/maskformer2_swin_tiny_bs16_50ep.yaml', resume=False, eval_only=False, num_gpus=2, num_machines=1, machine_rank=0, dist_url='tcp://127.0.0.1:49152', opts=['MODEL.WEIGHTS', '/kaggle/working/Mask2Former/weights/swin_tiny_patch4_window7_224.pkl', 'OUTPUT_DIR', '/kaggle/working/output', 'SOLVER.IMS_PER_BATCH', '4', 'INPUT.MIN_SIZE_TRAIN', '(640,)', 'INPUT.MAX_SIZE_TRAIN', '640', 'INPUT.MIN_SIZE_TEST', '640', 'INPUT.MAX_SIZE_TEST', '640'])
[05/26 11:30:43 detectron2]: Rank of current process: 0. World size: 2
[05/26 11:30:44 detectron2]: Environment info:
-------------------------------  -----------------------------------------------------------------
sys.platform                     linux
Python                           3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy                            2.0.2
detectron2                       0.6 @/usr/local/lib/python3.12/dist-packages/detectron2
Compiler    